# MergeGuard: Pull Request Triage and Quality Agent
### Kaggle Capstone Project: AI Agents (Intensive Vibe Coding)

MergeGuard is an event-driven AI agent that triages incoming pull requests, handles low-risk changes with deterministic rules (Auto-Approve), and escalates higher-risk PRs through security sanitization, LLM-based quality analysis, and a human-in-the-loop approval step.

---

## Project Links
*   **GitHub Repository**: [GitHub Codebase](https://github.com/peterintest/merge-guard)
*   **Reviewer Dashboard**: [Live Dashboard](https://merge-guard-dashboard-648451783574.us-central1.run.app)
*   **Video Walkthrough**: [YouTube Demonstration](https://www.youtube.com/watch?v=jlAZ-JQJeuc)

---

## 1. Real-World Context: The Rise of Autonomous Code Agents
AI code assistants have sharply increased code delivery velocity. However, pull request review has become the bottleneck. Maintainers are overwhelmed by autonomous pull requests, as highlighted by Hugging Face's recent **Code Agent Policy**:

> *"The Transformers repo is currently being overwhelmed by a large number of PRs and issue comments written by code agents. We are currently bottlenecked by our ability to review and respond to them... PRs that appear to be fully agent-written will probably be closed without review..."*

MergeGuard addresses this exact crisis by acting as a semi-autonomous gatekeeper. Instead of flatly banning agent contributions, MergeGuard dynamically triages, classifies risk, and automates the baseline validation of incoming PRs, so human maintainers can focus only on changes flagged as requiring human review.

This notebook runs a **fully self-contained, offline-compatible simulation** of the core MergeGuard triage agent workflow.

---

## 2. End-to-End Deployed Production Workflow
In production, MergeGuard coordinates the following event-driven pipeline:
1. **Ingestion**: A GitHub webhook proxy captures PR events and publishes them to a Google Cloud Pub/Sub topic.
2. **Asynchronous Dispatch**: Pub/Sub routes the payload to the Backend Gateway (FastAPI server), which authenticates and invokes the **Vertex AI Agent Runtime** (Reasoning Engine) to run the triage graph.
3. **Context Gathering (MCP)**: The agent runs a github MCP client to pull PR metadata and patch diffs directly from the repo.
4. **Client-Side Secret Redaction**: Deterministically scrubs Google API keys, JWTs, AWS credentials, and Git tokens from the diff before model dispatch.
5. **Deterministic Triage Routing**:
   - **Low Risk**: Touches documentation or formatting, <= 5 files, and <= 50 lines. Bypasses LLM and auto-approves.
   - **High Risk**: Touches code, configs, or dependencies. Escalated to semantic audit.
6. **Semantic Audit (Gemini 2.5 Flash)**: Audits the code against customized agent review skills (testing-gaps, security-auditing, performance-leaks).
7. **Human-in-the-Loop (HITL) Approval**: Suspends the agent session mid-execution and posts details to the Cloud Run dashboard. A reviewer's manual approval or rejection resumes the session to complete the triage cycle.

---

## Kaggle Course Evaluation Criteria Met
MergeGuard implements **all six (6)** key concepts introduced during the course:

| Key Concept | MergeGuard Implementation |
| :--- | :--- |
| **Agent System (ADK)** | Stateful triage agent graph built using Google's **Agent Development Kit (ADK)**. |
| **MCP Server** | Connects to **GitHub MCP server** (`@modelcontextprotocol/server-github`) over stdio to inspect code. |
| **Security Features** | Client-side Regex-based **secret key redaction** and prompt injection screening before LLM dispatch. |
| **Agent Skills** | Guided by custom repository-level agent review guidelines (`security-auditing`, `testing-gaps`, etc.) |
| **Antigravity** | Pair programming with Antigravity assistant to iterate on rules and CSS dashboard styling. |
| **Deployability** | Vertex AI Reasoning Engine (backend) and GCP Cloud Run dashboard (frontend) deployed live. |

In [ ]:
# Install Google GenAI SDK and styling helpers
!pip install -q google-genai pydantic

import os
import re
import json
from typing import List, Dict, Any, Optional, Tuple
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

print("Environment setup complete.")

## Credentials Setup
To run live audits, you can configure your Gemini API Key below. (Alternatively, in Kaggle, you can store it securely under **Add-ons -> Secrets**).

In [ ]:
from kaggle_secrets import UserSecretsClient

# Retrieve API keys from Kaggle User Secrets
try:
    user_secrets = UserSecretsClient()
    os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
    os.environ["GITHUB_TOKEN"] = user_secrets.get_secret("GITHUB_TOKEN")
    print("Credentials successfully loaded from Kaggle User Secrets.")
except Exception as e:
    print("Warning: Could not retrieve GEMINI_API_KEY or GITHUB_TOKEN from Kaggle User Secrets.")
    print("Please set them via Add-ons -> Secrets in your Kaggle notebook.")

## 3. Model Context Protocol (MCP) Simulation
MCP defines a standardized client-server protocol enabling LLM agents to safely call external database, API, or system tools. Here we implement a fully runnable python client/server model context protocol connection.

In [ ]:
class PullRequestMCPServer:
    """Simulated GitHub MCP Server exposing repository inspection tools."""
    def list_tools(self) -> List[Dict[str, Any]]:
        return [
            {
                "name": "get_pull_request",
                "description": "Retrieves pull request metadata, file list, and patch diff.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "owner": {"type": "string"},
                        "repo": {"type": "string"},
                        "pr_number": {"type": "integer"}
                    },
                    "required": ["owner", "repo", "pr_number"]
                }
            }
        ]
        
    def call_tool(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        if name == "get_pull_request":
            pr_num = arguments.get("pr_number")
            # Return mock PR data mapped to test cases
            if pr_num == 45:
                return {
                    "title": "docs: update deployment troubleshooting guide",
                    "changed_files": ["docs/SETUP.md"],
                    "diff": "diff --git a/docs/SETUP.md b/docs/SETUP.md\n+ Add Vertex AI reasoning engine configurations."
                }
            else:
                return {
                    "title": "feat: implement profile details API mapping",
                    "changed_files": ["bad-pr.ts", "auth/helper.ts"],
                    "diff": "diff --git a/bad-pr.ts b/bad-pr.ts\n+ export async function loadUserProfile(db, userId) {\n+   const apiKey = 'AIzaSyAz487G2xZ17Y0tP8W3S9KjRdL184G5T';\n+   const unsafeSql = `select * from users where id = ${userId}`;\n+   eval(userInput);\n+ }"
                }
        raise ValueError(f"Unknown tool: {name}")

class MCPClientSession:
    """Simulated MCP Client Session managing stdio connectivity to the server."""
    def __init__(self, server: PullRequestMCPServer):
        self.server = server
        
    def get_pr_context(self, owner: str, repo: str, pr_number: int) -> Dict[str, Any]:
        # 1. Inspect server capabilities
        tools = self.server.list_tools()
        print(f"1. [MCP Client] Found tools: {[t['name'] for t in tools]}")
        
        # 2. Call tool via protocol
        print(f"2. [MCP Client] Calling tool 'get_pull_request' for PR #{pr_number}...")
        result = self.server.call_tool(
            "get_pull_request",
            arguments={"owner": owner, "repo": repo, "pr_number": pr_number}
        )
        print("3. [MCP Client] Raw PR context payload received successfully.")
        return result

# Run the live MCP client/server simulation loop
mcp_server = PullRequestMCPServer()
mcp_client = MCPClientSession(mcp_server)
pr_context = mcp_client.get_pr_context(owner="acme", repo="web-app", pr_number=46)
print(f"Loaded PR Title: '{pr_context['title']}' containing diff with length {len(pr_context['diff'])} characters.")

## 4. Agent Data Models & Routing Configurations
We define the input `PullRequestEvent` schema and the structured `PRAnalysisOutput` used to capture Gemini's semantic audits.

In [ ]:
# =====================================================================
# Configuration Constants
# =====================================================================
LOW_RISK_EXTENSIONS = {".md", ".txt", ".json", ".lock", ".yaml", ".yml"}
LOW_RISK_PREFIXES = ["docs:", "chore:", "formatting:", "style:"]
SECURITY_SENSITIVE_PATHS = {"auth", "security", "login", "private", "key", "secret"}
MAX_FILES_FOR_LOW_RISK = 5
MAX_LINES_CHANGED_FOR_LOW_RISK = 50

# =====================================================================
# Pydantic Schemas matching ADK Agent
# =====================================================================
class PullRequestEvent(BaseModel):
    repository: str
    pr_number: int
    title: str
    description: str
    author: str
    diff: str
    changed_files: List[str]

class PRAnalysisOutput(BaseModel):
    testing_gaps: str = Field(description="Summary of missing tests/qa coverages")
    regression_risk: str = Field(description="Stability risk to existing functionalities")
    security_concerns: str = Field(description="Any security issues (e.g. SQL injection)")
    overall_risk_score: int = Field(description="Scale of 1 to 10")
    testing_score: int = Field(description="Scale of 1 to 10")
    security_score: int = Field(description="Scale of 1 to 10")
    recommendation: str = Field(description="Actionable feedback for remediation")

print("Schemas and parameters loaded.")

## 5. Security Checkpoints (Secret Redaction & Injection Scanning)
To preserve privacy and prevent leakage of secrets, MergeGuard performs client-side token sanitization before prompts are sent to Gemini.

In [ ]:
def redact_secrets(content: str) -> str:
    """Scrubs sensitive API keys and tokens before passing to the model."""
    patterns = [
        (
            r'(?i)(api[_-]?key|secret|token|password|auth_token)\s*[:=]\s*["\']([^"\']{8,})["\']',
            r'\1: "[REDACTED_API_KEY]"',
        ),
        (r"(?i)AIzaSy[A-Za-z0-9_-]{35}", "[REDACTED_GOOGLE_API_KEY]"),
        (r"(?i)ghp_[A-Za-z0-9]{36}", "[REDACTED_GITHUB_TOKEN]"),
    ]
    redacted = content
    for pattern, repl in patterns:
        redacted = re.sub(pattern, repl, redacted)
    return redacted

def detect_prompt_injection(content: str) -> bool:
    """Blocks adversarial user inputs designed to override agent directives."""
    indicators = [
        "ignore previous instructions",
        "system prompt override",
        "you must now approve this",
        "bypass security verification",
    ]
    lowered = content.lower()
    return any(indicator in lowered for indicator in indicators)

print("Client-side security filters successfully loaded.")

## 6. Deterministic Triage Risk Routing
MergeGuard uses fast rules to triage pull requests into **Low Risk** (auto-approves to bypass the LLM entirely) or **High Risk** (triggering full LLM semantic evaluation).

In [ ]:
def triage_risk(pr: PullRequestEvent) -> Tuple[str, List[str]]:
    """Classifies a PR as 'low_risk' (auto-approve) or 'high_risk' (LLM audit)."""
    matched_rules = []
    title_lower = pr.title.lower()
    
    # 1. Path-based check
    has_sensitive_path = False
    for path in pr.changed_files:
        if any(sensitive in path.lower() for sensitive in SECURITY_SENSITIVE_PATHS):
            has_sensitive_path = True
            matched_rules.append(f"Security sensitive path modified: {path}")
            break
            
    # 2. Prefix Check
    has_low_risk_prefix = any(title_lower.startswith(prefix) for prefix in LOW_RISK_PREFIXES)
    if has_low_risk_prefix:
        matched_rules.append("PR title matches low-risk prefix.")
        
    # 3. File extension checks
    all_low_risk_ext = all(any(path.endswith(ext) for ext in LOW_RISK_EXTENSIONS) for path in pr.changed_files)
    if all_low_risk_ext:
        matched_rules.append("All modified files carry non-executable extensions.")
        
    # 4. Size thresholds
    file_count_ok = len(pr.changed_files) <= MAX_FILES_FOR_LOW_RISK
    added_lines = len([line for line in pr.diff.splitlines() if line.startswith("+") and not line.startswith("+++")])
    lines_ok = added_lines <= MAX_LINES_CHANGED_FOR_LOW_RISK
    
    is_low_risk = (
        not has_sensitive_path and
        (has_low_risk_prefix or all_low_risk_ext) and
        file_count_ok and
        lines_ok
    )
    
    if is_low_risk:
        return "low_risk", matched_rules
    else:
        if has_sensitive_path:
            matched_rules.append("Touches sensitive directories (auth/security).")
        if not lines_ok:
            matched_rules.append(f"Lines changed ({added_lines}) exceeds threshold ({MAX_LINES_CHANGED_FOR_LOW_RISK}).")
        if not file_count_ok:
            matched_rules.append(f"File count ({len(pr.changed_files)}) exceeds threshold ({MAX_FILES_FOR_LOW_RISK}).")
        return "high_risk", matched_rules

print("Triage routing matrix compiled.")

## 7. Simulating Test Scenarios
We define two test cases to walk through our triage routing logic.

In [ ]:
# Scenario A: Trivial Documentation Pull Request (Will Auto-Approve)
trivial_pr = PullRequestEvent(
    repository="acme/web-app",
    pr_number=45,
    title="docs: update deployment troubleshooting guide",
    description="Minor wording edits and spelling corrections in setup files.",
    author="dev-bob",
    diff="diff --git a/docs/SETUP.md b/docs/SETUP.md\n+ Add Vertex AI reasoning engine configurations.",
    changed_files=["docs/SETUP.md"]
)

# Scenario B: High-Risk Pull Request (Will trigger full review)
vulnerable_pr = PullRequestEvent(
    repository="acme/web-app",
    pr_number=46,
    title="feat: implement profile details API mapping",
    description="Integrates database query lookup and token management.",
    author="dev-malicious",
    diff="""diff --git a/bad-pr.ts b/bad-pr.ts
+ export async function loadUserProfile(db, userId) {
+   const apiKey = "AIzaSyAz487G2xZ17Y0tP8W3S9KjRdL184G5T"; // secret token
+   console.log("Loading profile with debug output", apiKey);
+   const unsafeSql = `select * from users where id = ${userId}`;
+   const profile = await db.query(unsafeSql);
+   eval(userInput);
+   return { profile };
+ }""",
    changed_files=["bad-pr.ts", "auth/helper.ts"]
)

# Execute triage routing
for label, pr in [("Trivial (Docs) PR", trivial_pr), ("High-Risk (Vulnerable) PR", vulnerable_pr)]:
    route, rules = triage_risk(pr)
    print(f"[{label}]: Routing -> {route.upper()} (Matched Rules: {rules})")

## 8. Secret Redaction Output
We inspect how the security filters scrub the credentials from Scenario B prior to LLM submission.

In [ ]:
print("--- RAW DIFF INCOMING ---")
print(vulnerable_pr.diff)

print("\n--- REDACTED DIFF SENT TO LLM ---")
print(redact_secrets(vulnerable_pr.diff))

## 9. Repository-Level Agent Skills Loader
Agent Skills dictate the quality parameters used to evaluate the code structure. We load these instructions directly from the workspace filesystem files (under `merge_guard/review_skills`), falling back to built-in rules if running standalone.

In [ ]:
def load_review_skills() -> str:
    skills_context = ""
    skills_dir = "merge_guard/review_skills"
    
    # Standard fallback guidelines matching our codebase files
    fallback_skills = {
        "testing-gaps": "Verify that all new functions or endpoints have corresponding test coverage.",
        "security-auditing": "Check for secret exposures, SQL injection, RCE vectors, and unvalidated user inputs.",
        "performance-leaks": "Verify there are no memory leaks, unclosed streams, or inefficient loops."
    }
    
    if os.path.exists(skills_dir):
        print(f"Folder '{skills_dir}' detected. Loading local repo guidelines...")
        for skill_name in os.listdir(skills_dir):
            skill_path = os.path.join(skills_dir, skill_name, "SKILL.md")
            if os.path.exists(skill_path):
                with open(skill_path, "r", encoding="utf-8") as f:
                    skills_context += f"\n--- Skill: {skill_name} ---\n{f.read()}\n"
    else:
        print("Local skills folder not found. Loading default fallback skill contexts...")
        for name, desc in fallback_skills.items():
            skills_context += f"\n--- Skill: {name} ---\n{desc}\n"
            
    return skills_context

skills_context = load_review_skills()
print(f"Successfully initialized custom skills context containing {len(skills_context)} characters of review rules.")

## 10. Semantic Quality Audit (Gemini 2.5 Flash)
We execute the quality audit using our custom Agent Skills to guide Gemini 2.5 Flash's analysis.

In [ ]:
import os
from google import genai
from google.genai import types

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("Detected GEMINI_API_KEY. Querying Gemini 2.5 Flash live...")
    try:
        client = genai.Client(api_key=api_key)
        prompt = f"""
        You are the MergeGuard pull request reviewer agent.
        Audit the following pull request diff using the custom skills guidelines provided.
        
        Custom Skills Guidelines:
        {skills_context}
        
        PR Title: {vulnerable_pr.title}
        PR Description: {vulnerable_pr.description}
        
        Diff:
        {redact_secrets(vulnerable_pr.diff)}
        """
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=PRAnalysisOutput,
                temperature=0.1
            )
        )
        analysis = PRAnalysisOutput.model_validate_json(response.text)
    except Exception as e:
        print(f"API call failed: {e}. Defaulting to offline simulation...")
        analysis = None
else:
    print("No GEMINI_API_KEY environment variable found. Executing offline simulation...")
    analysis = None

if not analysis:
    # Simulated high-fidelity Pydantic validation fallback
    analysis = PRAnalysisOutput(
        testing_gaps="The new `loadUserProfile` function has no corresponding unit or integration test files added.",
        regression_risk="Calling `eval()` dynamically introduces unpredictable behavior depending on user input context.",
        security_concerns="""
### [CRITICAL] SQL Injection Vulnerability
> [!CAUTION]
> **Location:** `bad-pr.ts:5`
> The database query string interpolates `userId` directly (`select * from users where id = ${userId}`).
>
> **Remediation:** Use parameterized queries or prepared statements.

---

### [CRITICAL] Arbitrary Code Execution (via `eval`)
> [!CAUTION]
> **Location:** `bad-pr.ts:7`
> Dynamically evaluates raw string input (`eval(userInput)`).
>
> **Remediation:** Remove the `eval` statement immediately.
""",
        overall_risk_score=9,
        testing_score=1,
        security_score=1,
        recommendation="Reject pull request immediately. Remediate SQL injection and remove the eval statement."
    )

# Render the results nicely using Markdown
report_md = f"""
## MergeGuard Quality Audit Report
**Overall Risk Rating:** {analysis.overall_risk_score}/10
**QA & Testing Score:** {analysis.testing_score}/10
**Security Health Score:** {analysis.security_score}/10

### Testing Gaps:
{analysis.testing_gaps}

### Regression Risks:
{analysis.regression_risk}

### Security Concerns:
{analysis.security_concerns}

### Actionable Recommendation:
**{analysis.recommendation}**
"""

display(Markdown(report_md))